# Research: Adaptive-Conformal-Risk

Adaptive Conformal Inference Risk Overlay on Multi-Factor Momentum. Implements the ACI algorithm (Gibbs & Candès, 2021) as a dynamic risk management overlay. ACI provides online-adjusted prediction intervals

**Calibration target**: `AdaptiveConformalRisk` in `main.py`

In [1]:
# Initialize QuantBook
from datetime import datetime
from QuantConnect.Research import QuantBook

qb = QuantBook()
symbols = {}

symbols['AAPL'] = qb.add_equity('AAPL', Resolution.DAILY).symbol
symbols['MSFT'] = qb.add_equity('MSFT', Resolution.DAILY).symbol
symbols['NVDA'] = qb.add_equity('NVDA', Resolution.DAILY).symbol
symbols['BAC'] = qb.add_equity('BAC', Resolution.DAILY).symbol
symbols['AMZN'] = qb.add_equity('AMZN', Resolution.DAILY).symbol

start = datetime(2020, 1, 1)
end = datetime(2026, 1, 1)
history = qb.history([symbols['AAPL'], symbols['MSFT'], symbols['NVDA'], symbols['BAC'], symbols['AMZN']], start, end, Resolution.DAILY)
print(f"History: {len(history)} rows")
if len(history) > 0:
    display(history.head())

History: 3016 rows


close       high        low       open  \
symbol time                                                              
AAPL   2020-01-02 16:00:00  74.333516  74.395389  73.056470  73.323759   
       2020-01-03 16:00:00  73.610847  74.390439  73.380681  73.492052   
       2020-01-06 16:00:00  74.197397  74.236995  72.452595  72.687710   
       2020-01-07 16:00:00  73.848437  74.469636  73.623221  74.241945   
       2020-01-08 16:00:00  75.036387  75.345749  73.541549  73.546499   

                                 volume  
symbol time                              
AAPL   2020-01-02 16:00:00  130837932.0  
       2020-01-03 16:00:00  131727294.0  
       2020-01-06 16:00:00  113043192.0  
       2020-01-07 16:00:00  102025531.0  
       2020-01-08 16:00:00  126341934.0

## 1. Data Exploration

Statistical overview: returns distribution, correlations, volatility.

In [2]:
import pandas as pd
import numpy as np

if len(history) == 0:
    print('WARNING: No history data available')
    closes = pd.DataFrame()
    returns = pd.DataFrame()
else:
    if isinstance(history.index, pd.MultiIndex):
        closes = history['close'].unstack(level=0)
    else:
        closes = history['close']
    returns = closes.pct_change().dropna()
    print("=== Returns Summary ===")
    print(returns.describe().round(4))
    print()
    print("=== Correlation Matrix ===")
    print(returns.corr().round(3))

=== Returns Summary ===
symbol       AAPL        BAC
count   1507.0000  1507.0000
mean       0.0004     0.0002
std        0.0128     0.0155
min       -0.1286    -0.1540
25%        0.0000     0.0000
50%        0.0000     0.0000
75%        0.0000     0.0000
max        0.1198     0.1780

=== Correlation Matrix ===
symbol   AAPL    BAC
symbol              
AAPL    1.000  0.493
BAC     0.493  1.000


## 2. Signal Analysis

Evaluate momentum signals across multiple timeframes.

In [3]:
# Signal computation
if len(returns) == 0:
    print('No data for signal analysis')
    signal = pd.Series(dtype=float)
else:
    # Compute momentum signals
    mom_12m = closes.pct_change(252).dropna()
    mom_6m = closes.pct_change(126).dropna()
    mom_3m = closes.pct_change(63).dropna()
    signal = mom_12m.iloc[:, 0] if mom_12m.shape[1] > 0 else mom_12m
    signal.name = 'momentum_12m'

print("=== Signal Statistics ===")
if len(signal) > 0:
    print(f"Signal mean: {signal.mean():.6f}")
    print(f"Signal std: {signal.std():.6f}")
    print(f"Signal range: [{signal.min():.6f}, {signal.max():.6f}]")

=== Signal Statistics ===
Signal mean: 0.074265
Signal std: 0.216628
Signal range: [-0.145484, 1.201540]


## 3. Parameter Sensitivity

Test different parameter values for lookback.

In [4]:
# Parameter sensitivity scan
if len(returns) == 0:
    print('No data for parameter scan')
    results = []
    best_params = 'N/A'
else:
    # Generic parameter scan
    lookbacks = [10, 20, 30, 60]
    results = []
    for lb in lookbacks:
        vol = closes.iloc[:, 0].pct_change().rolling(lb).std().mean()
        results.append((f'lookback={lb}', vol))
    best_params = min(results, key=lambda x: x[1])[0]

print("=== Parameter Scan Results ===")
for params, metric in results:
    print(f"  {params}: metric={metric:.4f}")
print(f"\nBest: {best_params}")

=== Parameter Scan Results ===
  lookback=10: metric=0.0052
  lookback=20: metric=0.0053
  lookback=30: metric=0.0054
  lookback=60: metric=0.0053

Best: lookback=10


## 4. Calibration to main.py

Mapping research findings to algorithm parameters:

| Parameter | Research Value | main.py Default |
|-----------|---------------|----------------|
| lookback | (optimal) | default |

In [5]:
# Calibration summary
calibration = {
    "lookback": "<research_value>",
}

print("=== Calibration Parameters ===")
for k, v in calibration.items():
    print(f"  {k}: {v}")
print("\nTo apply: update get_parameter() defaults in main.py")

=== Calibration Parameters ===
  lookback: <research_value>

To apply: update get_parameter() defaults in main.py


## 5. Conclusion & Next Iteration

**Findings:**
- Universe (AAPL, MSFT, NVDA, BAC, AMZN) analyzed with momentum signals
- Parameter sensitivity for lookback scanned

**Next iteration:**
- Test alternative universe composition
- Add regime-conditional analysis
- Validate against backtest results in main.py